# 🧬 Bonus Activity — De Novo Binder Design with RFdiffusion
### Same three teams, one step beyond the main workshop.

In the main workshop you **redesigned** an existing protein — you gave ProteinMPNN a shape and it wrote a new sequence for the same shape. That's inverse folding (2022).

Now you'll **generate entirely new proteins from scratch** — small binders that grip a specific face of your team's target. RFdiffusion starts from pure noise and denoises into a protein-like backbone that satisfies your constraints. This is the 2023–2026 generative frontier, and it's what Xaira, Generate, and Isomorphic do at commercial scale.

| Team | Target | What your binder could be used for |
|---|---|---|
| **PETase** | Plastic depolymerase | Thermostabilizer (keeps the enzyme folded at industrial temperatures) or substrate-recruiter (localizes PETase to PET surfaces) |
| **ZAR1** | Plant immune receptor | Effector mimic that activates plant immunity — potential crop protection lead |
| **CarRP** | Carotenoid bifunctional enzyme | Localization / channeling partner for engineered biomanufacturing strains (co-localize upstream and downstream carotenoid enzymes) |

All three are legitimate, non-biomedical applications of modern binder design — a glimpse of what the industrial and agricultural side of the field could look like once it catches up with the therapeutic side.

**What you'll do:**
1. Load your team's target (same one as the main workshop)
2. Generate a binder backbone with RFdiffusion (~15 min)
3. Design its sequence with ProteinMPNN
4. Validate with AlphaFold — does the binder fold AND grip the target correctly?
5. Visualize and compare across teams

---

## ✅ Step 1: Verify environment

*⏱️ Expected runtime: ~3 seconds.*


In [ ]:
import os, glob, torch
from colabdesign.mpnn import mk_mpnn_model
from colabdesign.af   import mk_af_model
import py3Dmol

assert torch.cuda.is_available(), "GPU required"
assert os.path.isdir(os.path.expanduser("~/RFdiffusion")), "RFdiffusion not installed"

af_params = glob.glob(os.path.expanduser("~/params/**/params_model_1.npz"), recursive=True)
assert af_params, "AF2 params missing under ~/params"

print(f"✅ Environment ready. GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# =============================================================================
# PICK YOUR TEAM
# =============================================================================
# Edit this ONE line, then run all cells. The dictionary in the next cell
# uses TEAM to select your protein, hotspots, and narrative.
# =============================================================================

TEAM = "PETase"   # <-- "PETase", "ZAR1", or "CarRP"


## 🎯 Step 2: Confirm your team's target

**Use the same team you used in the main workshop** so the debrief compares apples to apples.

Make sure `TEAM` is set correctly in the picker cell above before running the cell below.


In [ ]:
# =============================================================================
# THE THREE TEAM TARGETS
# =============================================================================


TARGETS = {
    "PETase": {
        "label":      "PETase (easy / industrial)",
        "source":     "pdb",
        "pdb_id":     "6EQE",
        "chain":      "A",
        "range":      None,                     # use full chain
        "organism":   "Ideonella sakaiensis (bacterium)",
        "function":   "Plastic (PET) depolymerase",
        "hotspots":   "A159,A161,A185",         # catalytic-face residues for binder design
        "why":        (
            "The 'easy' case. Well-characterized, single-domain α/β hydrolase (~290 aa). "
            "Plenty of training data from related cutinases and esterases. This is the "
            "protein-design equivalent of a clean test set — AI models should look good here."
        ),
        "preamble_tie": (
            "Industrial biomanufacturing, slide 10. PETase is the poster child for "
            "designed enzymes in the circular bioeconomy — FAST-PETase was engineered "
            "via ML in 2022. Your team will feel what protein design looks like when "
            "the data is good and the problem is tractable."
        ),
    },
    "ZAR1": {
        "label":      "ZAR1 (plant immune receptor / hard case)",
        "source":     "pdb",
        "pdb_id":     "6J5T",
        "chain":      "A",
        "range":      (1, 200),                 # truncate to CC + NB-ARC (~200 aa)
        "organism":   "Arabidopsis thaliana (plant)",
        "function":   "NLR immune receptor — detects bacterial effectors, forms pentameric 'resistosome'",
        "hotspots":   "A14,A17,A24",            # CC-domain surface — proposed effector face
        "why":        (
            "The 'hard' case. Plant immune receptors are large, multidomain, and form "
            "dynamic oligomeric complexes (the ZAR1 resistosome is a pentamer that "
            "punctures cell membranes). Plant proteins are significantly under-represented "
            "in training sets vs. bacterial/human proteins. We truncate to the CC + NB-ARC "
            "domain (~200 aa) to keep the workshop tractable — but the full biology is "
            "exactly the kind of thing current models struggle with."
        ),
        "preamble_tie": (
            "Limitations slide (generalization + dynamics). ZAR1 exercises two failure "
            "modes at once — it's a plant protein (training-data asymmetry) AND it's a "
            "conformationally dynamic oligomer (static-structure bias). Your team will feel "
            "the edges of current capability."
        ),
    },
    "CarRP": {
        "label":      "CarRP (Schmidt Sciences relevant / industrial fungal)",
        "source":     "afdb",                   # AlphaFold Database — no experimental structure
        "uniprot":    "Q9UUQ6",
        "chain":      "A",
        "range":      (1, 330),                 # R domain = lycopene cyclase (functional alone)
        "organism":   "Mucor circinelloides (fungus)",
        "function":   "Bifunctional lycopene cyclase + phytoene synthase — carotenoid biosynthesis",
        "hotspots":   "A45,A80,A150",           # cyclase-domain face (approximate)
        "why":        (
            "The 'real world' case. CarRP is a fungal bifunctional enzyme with no "
            "experimental structure — we use an AlphaFold Database prediction. "
            "This is what most industrially-relevant targets actually look like when "
            "you try to design on them. Fungi are under-represented in training data. "
            "The bifunctional architecture (lycopene cyclase + phytoene synthase fused "
            "into one polypeptide) is rare and challenging."
        ),
        "preamble_tie": (
            "Industrial biomanufacturing (slide 10) + the PLAID-Bio thesis (data slide, "
            "slide 14). Carotenoids are a ~$1.8B global market — CarRP sits at the heart "
            "of microbial lycopene/β-carotene production. Your team will feel what "
            "protein design is like on a real non-model-organism industrial target."
        ),
    },
}

if TEAM not in TARGETS:
    raise ValueError(f"Unknown TEAM '{TEAM}'. Options: {list(TARGETS)}")

target = TARGETS[TEAM]
print("=" * 70)
print(f"🧬 Team {TEAM}: {target['label']}")
print("=" * 70)
print(f"Organism:   {target['organism']}")
print(f"Function:   {target['function']}")
print()
print(f"Why this protein:")
print(f"  {target['why']}")
print()
print(f"Preamble tie-in:")
print(f"  {target['preamble_tie']}")
print("=" * 70)

## 📥 Step 3: Load the target

*⏱️ Expected runtime: ~5–10 seconds.*


In [ ]:
# Load the target protein structure
import os, subprocess

os.makedirs("inputs", exist_ok=True)
input_pdb = f"inputs/{TEAM}.pdb"

def _download(url: str, dest: str, what: str):
    """Download with explicit error checking. wget -q hides 404s."""
    result = subprocess.run(f"wget -q {url} -O {dest}", shell=True)
    if result.returncode != 0 or not os.path.isfile(dest) or os.path.getsize(dest) < 1000:
        size = os.path.getsize(dest) if os.path.isfile(dest) else 0
        raise RuntimeError(
            f"Failed to download {what} from {url}\n"
            f"  exit={result.returncode}, size={size} bytes\n"
            f"  Check internet connectivity and that the resource still exists."
        )

if target["source"] == "pdb":
    url = f"https://files.rcsb.org/download/{target['pdb_id']}.pdb"
    _download(url, input_pdb, f"PDB {target['pdb_id']}")
    print(f"✅ Downloaded PDB {target['pdb_id']} from RCSB")
elif target["source"] == "afdb":
    url = f"https://alphafold.ebi.ac.uk/files/AF-{target['uniprot']}-F1-model_v4.pdb"
    _download(url, input_pdb, f"AlphaFold model for {target['uniprot']}")
    print(f"✅ Downloaded AlphaFold prediction for UniProt {target['uniprot']} (CarRP)")
    print("   Note: This is a PREDICTED structure, not an experimental one.")
    print("         Many industrial targets only have AlphaFold predictions available.")

# Truncate if a range is specified (keep only the relevant domain)
if target["range"] is not None:
    lo, hi = target["range"]
    trunc = f"inputs/{TEAM}_trunc.pdb"
    with open(input_pdb) as fin, open(trunc, "w") as fout:
        for line in fin:
            if line.startswith(("ATOM", "HETATM")):
                # Strict chain filter — drop anything not on our target chain.
                # (Old logic let through chain " " which leaked solvent into trunc.)
                if line[21] != target["chain"]:
                    continue
                try:
                    resnum = int(line[22:26])
                    if resnum < lo or resnum > hi:
                        continue
                except ValueError:
                    pass
            fout.write(line)
    input_pdb = trunc
    print(f"✂️  Truncated to chain {target['chain']} residues {lo}-{hi}")

# Report size
n_res = len({line[22:26] for line in open(input_pdb)
             if line.startswith("ATOM") and line[21] == target["chain"]})
print(f"📏 Working structure: {n_res} residues")


## 🎲 Step 4: Generate a binder backbone with RFdiffusion

This is the generative step. RFdiffusion takes:
- Your target structure (fixed — we're not changing it)
- A set of **hotspot residues** — the patch on the target where the binder should land
- A desired binder length (~70 residues here)

And it outputs a small protein backbone that sits on those hotspots.

*⏱️ Expected runtime: **10–15 minutes** (≈50 diffusion timesteps on the A10G GPU). This is the slowest cell in either notebook — don't worry if you don't see output for a while.*


In [ ]:
import subprocess, random, string, time

BINDER_LENGTH = 70   # residues in the designed binder

run_name = f"{TEAM}_bind_" + "".join(random.choices(string.ascii_lowercase, k=4))
output_prefix = f"outputs/{run_name}"
os.makedirs("outputs", exist_ok=True)

# Contig: target chain (all residues, fixed) / 0 (chain break) / new binder
contigs = f"{target['chain']}1-999/0 {BINDER_LENGTH}-{BINDER_LENGTH}"

rfd_home = os.path.expanduser("~/RFdiffusion")
entry = os.path.join(rfd_home, "scripts", "run_inference.py")
if not os.path.isfile(entry):
    entry = os.path.join(rfd_home, "run_inference.py")

cmd = (
    f"python {entry} "
    f"inference.output_prefix={output_prefix} "
    f"inference.num_designs=1 "
    f"inference.input_pdb={input_pdb} "
    f"'contigmap.contigs=[{contigs}]' "
    f"'ppi.hotspot_res=[{target["hotspots"]}]'"
)

print(f"🎲 Generating a binder against team {TEAM}'s target hotspots: {target['hotspots']}")
print(f"$ {cmd}\n")

t0 = time.time()
proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
t1 = time.time()

if proc.returncode != 0:
    # RFdiffusion errors mostly land in stderr; print both to be safe.
    print("❌ RFdiffusion failed.")
    print("--- STDOUT (last 1000 chars) ---")
    print(proc.stdout[-1000:])
    print("--- STDERR (last 1500 chars) ---")
    print(proc.stderr[-1500:])
    raise RuntimeError("RFdiffusion failed")

# RFdiffusion writes its tqdm progress to stderr; show both for transparency.
print("--- STDOUT tail ---")
print(proc.stdout[-500:])
print("--- STDERR tail (RFdiffusion progress) ---")
print(proc.stderr[-500:])
print(f"\n✅ Backbone generated in {t1-t0:.0f}s")

binder_backbones = sorted(f for f in os.listdir("outputs")
                          if f.startswith(run_name) and f.endswith(".pdb"))
binder_backbone = f"outputs/{binder_backbones[0]}"
print(f"📦 {binder_backbone}")


## ✍️ Step 5: Design the binder's sequence

Same tool as the main workshop (ProteinMPNN) — but now we fix the target's sequence (we're not redesigning your protein) and only design the binder chain.

*⏱️ Expected runtime: ~10 seconds.*


In [ ]:
mpnn = mk_mpnn_model()
mpnn.prep_inputs(pdb_filename=binder_backbone, chain="A,B", fix_pos="A", rm_aa="C")

out = mpnn.sample(num=2, temperature=0.1)
binder_designs = []
for i, seq in enumerate(out["seq"]):
    binder_seq = seq.split("/")[-1]   # binder is the second chain
    score = float(out["score"][i])
    binder_designs.append({"idx": i, "sequence": binder_seq, "score": score})
    print(f"  Design {i+1}: score={score:.3f}  len={len(binder_seq)}")
    print(f"    {binder_seq}")

print(f"\n✅ Team {TEAM}: {len(binder_designs)} binder sequence(s) designed.")


## 🔬 Step 6: Validate the complex with AlphaFold

Does the binder fold correctly AND still grip the target?

This is a tougher test than the main workshop. We need:
- **Target** stays folded (it was folded to start with, so this should be fine)
- **Binder** folds as designed
- **Interface** is preserved — binder still contacts the hotspot residues

> ℹ️ Reminder: ColabDesign reports pLDDT and ipTM on a **0–1 scale**. ipTM > 0.5 is the conventional threshold for "plausible interface."

*⏱️ Expected runtime: ~1–2 minutes (2 binders × ~30s each).*


In [ ]:
af = mk_af_model(protocol="binder", use_templates=False, num_recycles=3)
complex_results = []

for d in binder_designs:
    af.prep_inputs(pdb_filename=binder_backbone, chain="A,B")
    # full complex sequence: target fixed + designed binder
    # (colabdesign handles this internally when protocol='binder')
    af.set_seq(binder_len=len(d["sequence"]), seq=d["sequence"])
    af.predict(num_recycles=3, verbose=False)

    plddt = float(af.aux["log"]["plddt"])
    iptm  = float(af.aux["log"].get("iptm", -1))   # interface pTM — critical for binders
    pae   = float(af.aux["log"].get("pae", -1))
    rmsd  = float(af.aux["log"].get("rmsd", -1))

    d.update({"plddt": plddt, "iptm": iptm, "rmsd": rmsd, "pae": pae})

    af_path = f"outputs/{TEAM}_binder{d['idx']}_af.pdb"
    af.save_pdb(af_path)
    d["af_pdb"] = af_path

    verdict = "✅ plausible complex" if iptm > 0.5 else "⚠️  weak interface"
    print(f"  Binder {d['idx']+1}: pLDDT={plddt:.2f}  ipTM={iptm:.2f}  PAE={pae:.2f}   {verdict}")
    complex_results.append(d)


## 👁️ Step 7: Visualize your binder against your target

- **Gray** = your team's target (PETase / ZAR1 / CarRP)
- **Colored** = the de novo designed binder (blue = high pLDDT, red = low)
- Look for: does the binder sit on the hotspot residues? Does it look like a compact, stable fold?

In [ ]:
d = complex_results[0]

print(f"Team {TEAM} — De novo binder")
print(f"  pLDDT: {d['plddt']:.2f}  ipTM: {d['iptm']:.2f}")
print(f"  Binder sequence: {d['sequence']}")

with open(d["af_pdb"]) as f: complex_pdb = f.read()

view = py3Dmol.view(width=900, height=500)
view.addModel(complex_pdb, "pdb")
# Target = chain A in gray
view.setStyle({"chain": "A"}, {"cartoon": {"color": "lightgray"}})
# Binder = chain B colored by pLDDT
view.setStyle({"chain": "B"}, {"cartoon": {"colorscheme": {"prop": "b",
                                                           "gradient": "roygb",
                                                           "min": 50, "max": 90}}})
# Hotspot residues on target, shown as sticks in amber
for h in target["hotspots"].split(","):
    resnum = int(h.strip()[1:])  # skip chain letter
    view.addStyle({"chain": "A", "resi": resnum},
                  {"stick": {"color": "orange"}})
view.zoomTo()
view.show()


## 🗣️ Step 8: Final debrief — three teams, two activities

Compare across teams, now with both activities in hand:

|  | PETase | ZAR1 | CarRP |
|---|---|---|---|
| Main workshop (ProteinMPNN + AF2) | ? | ? | ? |
| Bonus (RFdiffusion binders + AF2 ipTM) | ? | ? | ? |

Questions for the room:
1. **Which team's binder looks most credible?** (Probably PETase.)
2. **Which team's binder is borderline or weak?** (Probably CarRP or ZAR1.)
3. **Why?** — All three used the same tools. The difference is entirely in the target protein's representation in training data and in the difficulty of the binding surface.

### The thesis this workshop demonstrates

Today's protein design tools are extraordinary but **biased toward the data they were trained on**. That data is biomedical-heavy, bacterial-heavy, and structure-rich. Industrial, agricultural, environmental, and non-model-organism targets are under-represented — and you just felt that asymmetry directly.

Closing that gap is not a modeling problem. It's a **data problem**: generating the sequence-structure-function measurements for under-represented biology at scale, so the next generation of models has something to train on.

That's what programs like PLAID-Bio are designed to do. You just ran a miniature version of the future that investment enables.

---

### References
- **RFdiffusion** — Watson et al., *Nature* (2023)
- **ProteinMPNN** — Dauparas et al., *Science* (2022)
- **AlphaFold2** — Jumper et al., *Nature* (2021)
- **ZAR1 resistosome** — Wang et al., *Science* (2019)
- **CarRP** — Velayos et al., *Eur. J. Biochem.* (2000)
- **PETase / FAST-PETase** — Tournier et al., *Nature* (2020); Lu et al., *Nature* (2022)
